# Final Model Training

## Overview
This notebook demonstrates the loading of the 60-feature dataset and the exact training process of the final `CatBoostRegressor` using the optimal hyperparameters found during Optuna tuning.

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
import json
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data Checkpoints
We load the preprocessed and engineered Parquet datasets.

In [2]:
X_train = pd.read_parquet('../checkpoints/X_train_cat.parquet')
X_test = pd.read_parquet('../checkpoints/X_test_cat.parquet')
y_train = np.load('../checkpoints/y_train.npy')
y_test = np.load('../checkpoints/y_test.npy')

print(f'Train size: {X_train.shape} | Test size: {X_test.shape}')

Train size: (211683, 60) | Test size: (35614, 60)


## 2. Load Hyperparameters
We load the optimal hyperparameters discovered from 200 trials of Bayesian optimization.

In [3]:
with open('../models/best_params.json', 'r') as f:
    best_params = json.load(f)['CatBoost']
print(f'Optimal hyperparameters: {json.dumps(best_params, indent=4)}')

Optimal hyperparameters: {
    "iterations": 3786,
    "depth": 8,
    "learning_rate": 0.07332399202862067,
    "l2_leaf_reg": 23.099367373513576,
    "subsample": 0.9200116523638655,
    "random_strength": 0.41053429686374326,
    "bagging_temperature": 3.5987443865985447,
    "border_count": 207,
    "min_data_in_leaf": 96
}


## 3. Train Production Model
We define the categorical features exactly as expected by CatBoost, and train the model. Note: CatBoost native categorical handling handles all non-numeric features without requiring prior one-hot encoding.

In [4]:
CATBOOST_CAT = ['season', 'crop', 'state_name', 'crop_season', 'state_crop', 'geo_cluster_10', 'geo_cluster_30', 'geo_cluster_50']

model = CatBoostRegressor(**best_params, cat_features=CATBOOST_CAT, random_seed=42, verbose=500)
# We log-transform y_test on the fly for the evaluation set to match y_train
model.fit(X_train, y_train, eval_set=(X_test, np.log1p(y_test)))

# Note: In the production codebase, the final model is saved to '../models/CatBoost_best_final.cbm'
print('Training successfully completed.')

0:	learn: 0.8816716	test: 0.8955079	best: 0.8955079 (0)	total: 650ms	remaining: 41m 1s


500:	learn: 0.1972814	test: 0.3135864	best: 0.3135864 (500)	total: 4m 10s	remaining: 27m 24s


1000:	learn: 0.1851212	test: 0.3115281	best: 0.3114996 (994)	total: 8m 1s	remaining: 22m 20s


1500:	learn: 0.1776756	test: 0.3107375	best: 0.3107258 (1498)	total: 10m 16s	remaining: 15m 38s


2000:	learn: 0.1725747	test: 0.3106134	best: 0.3104577 (1873)	total: 12m 18s	remaining: 10m 58s


2500:	learn: 0.1676293	test: 0.3105090	best: 0.3104577 (1873)	total: 14m 13s	remaining: 7m 18s


3000:	learn: 0.1635142	test: 0.3107788	best: 0.3104456 (2775)	total: 16m 4s	remaining: 4m 12s


3500:	learn: 0.1591204	test: 0.3108210	best: 0.3104456 (2775)	total: 18m 11s	remaining: 1m 28s


3785:	learn: 0.1569816	test: 0.3108210	best: 0.3104456 (2775)	total: 19m 18s	remaining: 0us

bestTest = 0.3104455747
bestIteration = 2775

Shrink model to first 2776 iterations.


Training successfully completed.
